# 🛰️ Field Shift: Step 1 - Big Data Collection Pipeline
**NASA Space Apps Challenge 2026**

This notebook handles the extraction of 100,000+ data points by integrating:
1. **Google Earth Engine (GEE):** For verified agricultural coordinates & MODIS data (NDVI, LST).
2. **NASA POWER API:** For daily meteorological data.
3. **ISRIC SoilGrids:** For soil composition (Sand/Clay percentages).

In [ ]:
# --- 1. Environment Setup & Authentication ---
import os
import time
import requests
import pandas as pd
import numpy as np
import ee
from google.colab import drive

# Mount Google Drive for safe saving
drive.mount('/content/drive')

# Authenticate and Initialize Earth Engine
# ee.Authenticate() # Uncomment if running for the first time
ee.Initialize(project='mystic-impulse-513920-q5')
print("✅ Earth Engine is Authenticated and Initialized!")

In [ ]:
# --- 2. Generating Verified Agricultural Points (Geospatial Verification) ---
print("🔍 Generating 150 VERIFIED agricultural points across Egypt...")

# Define Egypt's borders & Get Land Cover Data
egypt = ee.FeatureCollection("FAO/GAUL/2015/level0").filter(ee.Filter.eq('ADM0_NAME', 'Egypt')).geometry()
landcover = ee.ImageCollection("ESA/WorldCover/v200").first().clip(egypt)

# Stratified Sampling to FORCE exactly 150 points inside Class 40 (Cropland)
points = landcover.stratifiedSample(
    numPoints=0, classBand='Map', region=egypt, scale=250,
    classValues=[40], classPoints=[150], geometries=True
)

features = points.getInfo()['features']
verified_locations = []
for i, f in enumerate(features):
    lon, lat = f['geometry']['coordinates']
    verified_locations.append({"name": f"Agri_Point_{i+1}", "lat": round(lat, 5), "lon": round(lon, 5)})

print(f"✅ Successfully generated {len(verified_locations)} VERIFIED coordinates!\n")

In [ ]:
# --- 3. Starting the BIG DATA Pipeline (~100k Rows) ---
print("🚀 Starting Data Extraction from NASA POWER, MODIS, and SoilGrids...")

master_df_list = []
checkpoint_file = '/content/drive/MyDrive/Field_Shift_NASA_Project/Field_Shift_Perfect_100k.csv'

# Ensure the directory exists
os.makedirs(os.path.dirname(checkpoint_file), exist_ok=True)

for i, loc in enumerate(verified_locations):
    loc_name = loc["name"]
    lat = loc["lat"]
    lon = loc["lon"]
    print(f"[{i+1}/150] 📍 Processing {loc_name}...")

    # A. NASA POWER
    url_nasa = "https://power.larc.nasa.gov/api/temporal/daily/point"
    params_nasa = {"parameters": "T2M,RH2M,PRECTOTCORR,GWETROOT", "community": "AG", "longitude": lon, "latitude": lat, "start": "20250101", "end": "20261031", "format": "JSON"}

    try:
        response_nasa = requests.get(url_nasa, params=params_nasa, timeout=15)
        if response_nasa.status_code == 200:
            df_loc = pd.DataFrame(response_nasa.json()['properties']['parameter'])
            df_loc.index = pd.to_datetime(df_loc.index, format='%Y%m%d')
            df_loc.index.name = 'Date'
            df_loc.reset_index(inplace=True)
            df_loc.rename(columns={'T2M': 'Temperature_Avg', 'RH2M': 'Relative_Humidity', 'PRECTOTCORR': 'Precipitation', 'GWETROOT': 'Root_Soil_Moisture'}, inplace=True)
        else:
            continue
    except:
        continue

    # B. Earth Engine (NDVI & LST)
    try:
        poi = ee.Geometry.Point([lon, lat])

        # NDVI
        modis_ndvi = ee.ImageCollection("MODIS/061/MOD13Q1").filterBounds(poi).filterDate('2025-01-01', '2026-10-31')
        def extract_ndvi(image):
            return ee.Feature(None, {'Date': image.date().format('YYYY-MM-dd'), 'NDVI': image.select('NDVI').reduceRegion(reducer=ee.Reducer.first(), geometry=poi, scale=250).get('NDVI')})
        df_ndvi = pd.DataFrame([{'Date': f['properties'].get('Date'), 'NDVI': f['properties'].get('NDVI')} for f in modis_ndvi.map(extract_ndvi).getInfo()['features']])
        df_ndvi['Date'] = pd.to_datetime(df_ndvi['Date'])
        df_ndvi['NDVI'] = df_ndvi['NDVI'] * 0.0001

        # LST
        modis_lst = ee.ImageCollection("MODIS/061/MOD11A1").filterBounds(poi).filterDate('2025-01-01', '2026-10-31')
        def extract_lst(image):
            return ee.Feature(None, {'Date': image.date().format('YYYY-MM-dd'), 'LST': image.select('LST_Day_1km').reduceRegion(reducer=ee.Reducer.first(), geometry=poi, scale=1000).get('LST_Day_1km')})
        df_lst = pd.DataFrame([{'Date': f['properties'].get('Date'), 'LST': f['properties'].get('LST')} for f in modis_lst.map(extract_lst).getInfo()['features']])
        df_lst['Date'] = pd.to_datetime(df_lst['Date'])
        df_lst['LST_Celsius'] = np.where(df_lst['LST'].notna(), (df_lst['LST'] * 0.02) - 273.15, np.nan)
        df_lst.drop(columns=['LST'], inplace=True)

        # Merge EE Data
        df_loc = pd.merge(df_loc, df_ndvi, on='Date', how='left')
        df_loc = pd.merge(df_loc, df_lst, on='Date', how='left')
        df_loc['NDVI'] = df_loc['NDVI'].interpolate(method='linear').ffill().bfill()
        df_loc['LST_Celsius'] = df_loc['LST_Celsius'].interpolate(method='linear').ffill().bfill()
    except:
        continue

    # C. SoilGrids
    url_soil = f"https://rest.isric.org/soilgrids/v2.0/properties/query?lon={lon}&lat={lat}&property=sand&property=clay&depth=0-5cm"
    sand_pct, clay_pct = 40.0, 30.0
    try:
        response_soil = requests.get(url_soil, timeout=10)
        if response_soil.status_code == 200:
            for layer in response_soil.json().get('properties', {}).get('layers', []):
                mean_val = layer['depths'][0]['values']['mean']
                if mean_val is not None:
                    if layer['name'] == 'sand': sand_pct = mean_val / 10
                    elif layer['name'] == 'clay': clay_pct = mean_val / 10
    except:
        pass

    df_loc['Soil_Sand_pct'] = sand_pct
    df_loc['Soil_Clay_pct'] = clay_pct
    df_loc['Crop_Kc'] = 1.15
    df_loc['Location_Name'] = loc_name
    master_df_list.append(df_loc)

    # Auto-save every 10 locations
    if (i + 1) % 10 == 0 or (i + 1) == len(verified_locations):
        pd.concat(master_df_list, ignore_index=True).to_csv(checkpoint_file, index=False)

In [ ]:
# --- 4. Final Output ---
final_df = pd.concat(master_df_list, ignore_index=True)
print(f"\n✅ MISSION ACCOMPLISHED! Extracted {len(final_df)} rows.")
display(final_df.head())